# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the dataset using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL.

Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a DatasetMetadata object

print(f"{metadata.name}: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Keywords: {metadata.keywords}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, their fields, and IDs. All entities are referenced by their Croissant `@id`.

First, let's inspect the record sets defined in this dataset.

In [ ]:
# Display all RecordSets and their @id
record_sets = list(dataset.metadata.record_sets)
print(f"Number of record sets in dataset: {len(record_sets)}\n")
for i, rs in enumerate(record_sets, 1):
    print(f"Record Set {i}:\n  @id: {rs['@id']}\n  Name: {rs.get('name', '(no name)')}\n  Description: {rs.get('description', '(no description)')}")
    # List fields for this record set
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields and their @id:")
    for field in fields:
        if isinstance(field, dict):
            print(f"    - {field['@id']}: {field.get('name', field['@id'])}")
        else:
            print(f"    - {field}")
    print()

For demonstration, let's show a sample record from the **main record set**. Update the `RECORD_SET_ID` variable below with the actual `@id` you want to use (see printed list above).

In [ ]:
# Update the RECORD_SET_ID variable after inspecting the above output.
# We will try to automatically pick the first one for this notebook demonstration.
if record_sets:
    RECORD_SET_ID = record_sets[0]['@id']
    print(f"Using record set: {RECORD_SET_ID}\n")
    # Show up to two records as sample
    for i, rec in enumerate(dataset.records(record_set=RECORD_SET_ID)):
        print(f"Sample record {i+1}:")
        pprint.pprint(rec)
        if i >= 1:
            break
else:
    print("No record sets found in dataset.")

## 3. Data Extraction
Load data from specific record set(s) into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all dataframes for all record sets
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    print(f"Extracting records for record set: {rs_id}")
    rows = list(dataset.records(record_set=rs_id))
    if rows:
        df = pd.DataFrame(rows)
        print(f"  Columns ({len(df.columns)}): {df.columns.tolist()}")
        # Show top records
        print(df.head(2))
        dataframes[rs_id] = df
    else:
        print("  No records found for this record set.")
    print()

From the above, choose the main table for analysis. Usually, the main protein abundance table. Update the `MAIN_RECORD_SET_ID` accordingly (using the primary record set).

In [ ]:
MAIN_RECORD_SET_ID = RECORD_SET_ID  # you may overwrite if needed
main_df = dataframes[MAIN_RECORD_SET_ID].copy()
print(f"Main DataFrame columns: {main_df.columns.tolist()}")
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

First, let's identify numeric fields for analysis (e.g., abundance, MW, PeptideCount, or similar based on the data columns present).

In [ ]:
# Guess numeric fields: we'll take numeric columns for demonstration
numeric_fields = main_df.select_dtypes(include=['number']).columns.tolist()
print("Numeric fields in main record set:", numeric_fields)

# If there's a likely abundance or MW field, select that for filtering
# We'll try 'MW' (molecular weight) or the first available numeric if MW not present
candidate_fields = [name for name in numeric_fields if 'abundance' in name.lower() or 'mw' in name.lower() or 'peptide' in name.lower() or 'count' in name.lower()]
if candidate_fields:
    numeric_field = candidate_fields[0]
else:
    numeric_field = numeric_fields[0] if numeric_fields else None

if numeric_field:
    print(f"Using numeric field for demonstration: {numeric_field}\n")
else:
    print("No numeric field detected in main DataFrame.")

In [ ]:
# Begin EDA if a numeric field exists
if numeric_field:
    # Filter: Select records where the numeric_field > a threshold (use 10 or median)
    if main_df[numeric_field].dtype.name.startswith('float') or main_df[numeric_field].dtype.name.startswith('int'):
        thresh = main_df[numeric_field].median()
    else:
        thresh = 10

    filtered_df = main_df[main_df[numeric_field] > thresh]
    print(f"Filtered records where {numeric_field} > {thresh:.2f} (showing up to 5):")
    print(filtered_df[[numeric_field]].head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by 'description'/'Description'/'Accession' if available (choose as example)
    preferred_group_fields = [x for x in main_df.columns if x.lower() in ['description', 'accession', 'protein']]
    group_field = preferred_group_fields[0] if preferred_group_fields else None

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head(5))
else:
    print("No numeric field to analyze.")

## 5. Visualization
Visualize data distributions or relationships using simple plots. For example, explore the distribution of the chosen numeric field or top entries by normalized value.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[numeric_field].dropna(), bins=40, kde=True)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()

    if group_field:
        # Top N groups by mean value
        grouped_means = main_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
        plt.figure(figsize=(10, 6))
        sns.barplot(x=grouped_means.values, y=grouped_means.index)
        plt.xlabel(f"Mean {numeric_field}")
        plt.title(f"Top 10 {group_field} by mean {numeric_field}")
        plt.tight_layout()
        plt.show()
else:
    print("Skipping visualization – no numeric field to plot.")

## 6. Conclusion
- We loaded and explored the Mass Spectrometry dataset via its Croissant schema using `mlcroissant`.
- We inspected record sets, loaded the main data table, and identified numeric fields for analysis, such as abundance or molecular weight.
- Basic preprocessing (filtering, normalization, grouping) and distributions were demonstrated for EDA.

Further steps could include more advanced visualizations, cross-referencing proteins with public databases, or modeling tasks for proteomics research.